# Deepfake Module - Training and Evaluation

This notebook handles the fine-tuning of the DINOv2 model for landmark detection and evaluates the integrated `DeepfakeClassifier`.

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
from PIL import Image
import numpy as np
import os

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Landmarks Dataset

We use `zguo0525/google-landmarks-v2-mini` for fine-tuning.

In [2]:
dataset_name = "zguo0525/google-landmarks-v2-mini"
print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name)

num_classes = len(dataset['train'].features['label'].names)
print(f"Number of landmark classes: {num_classes}")
label_names = dataset['train'].features['label'].names

Loading dataset: zguo0525/google-landmarks-v2-mini


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/493M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/498M [00:00<?, ?B/s]

...

data/test-00000-of-00002.parquet:   0%|          | 0.00/384M [00:00<?, ?B/s]

data/test-00001-of-00002.parquet:   0%|          | 0.00/385M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20191 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6206 [00:00<?, ? examples/s]

Number of landmark classes: 3103


## 2. Fine-tune DINOv2 for Landmark Detection

In [3]:
model_name = "facebook/dinov2-base"
processor = AutoImageProcessor.from_pretrained(model_name)

def transform(examples):
    inputs = processor([img.convert("RGB") for img in examples["image"]], return_tensors="pt")
    inputs["labels"] = examples["label"]
    return inputs

train_dataset = dataset["train"].with_transform(transform)
if "test" in dataset:
    eval_dataset = dataset["test"].with_transform(transform)
else:
    ds_split = dataset["train"].train_test_split(test_size=0.1)
    train_dataset = ds_split["train"].with_transform(transform)
    eval_dataset = ds_split["test"].with_transform(transform)

model = AutoModelForImageClassification.from_pretrained(
    model_name, 
    num_labels=num_classes, 
    id2label={str(i): name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
    ignore_mismatched_sizes=True
).to(device)

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2ForImageClassification LOAD REPORT from: facebook/dinov2-base
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
training_args = TrainingArguments(
    output_dir="./dinov2-landmarks-finetuned",
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
    # use_mps_device is deprecated in newer transformers versions.
    # The Trainer will automatically use MPS if available and no CUDA is found.
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).astype(np.float32).mean().item()}

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["labels"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

print("Starting training... (This may take a long time)")
trainer.train()

Starting training... (This may take a long time)


Epoch,Training Loss,Validation Loss,Accuracy
1,8.068349,8.051341,0.000322
2,7.884821,7.917457,0.001611
3,7.654185,7.825461,0.002578


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7572, training_loss=7.97591911291736, metrics={'train_runtime': 3964.7684, 'train_samples_per_second': 15.278, 'train_steps_per_second': 1.91, 'total_flos': 4.99752739347899e+18, 'train_loss': 7.97591911291736, 'epoch': 3.0})

In [1]:
save_directory = "./dinov2-landmarks-finetuned"
print(f"Saving model to {save_directory}")
trainer.save_model(save_directory)
processor.save_pretrained(save_directory)
print("Model saved successfully!")

Saving model to ./dinov2-landmarks-finetuned


NameError: name 'trainer' is not defined

## 3. Test Integrated DeepfakeClassifier

In [1]:
from deepfake_classifier import DeepfakeClassifier

classifier = DeepfakeClassifier()

def test_inference(image_path):
    img = Image.open(image_path)
    results = classifier.predict(img)
    if results['deepfake_analysis']:
        print(f"Deepfake Confidence: {results['deepfake_analysis']['deepfake_confidence']}")
    print(f"Final Decision: {results['final_decision']}")
    return results

W0506 14:46:34.265000 36898 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading Face Forensics model: prithivMLmods/deepfake-detector-model-v1


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Loading Scene model: birder-project/rope_vit_reg4_b14_capi-places365


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading fine-tuned Landmark model from: ../dinov2-landmarks-finetuned


OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: '../dinov2-landmarks-finetuned'.